<a href="https://colab.research.google.com/github/anandita-3217/EAI-Assignments/blob/main/Assignment2/SE26MAID034_EAI_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1. When actions have different costs in a state based search space, an obvious choice is to use best-first search where the evaluation function is the cost of the path from the root to the current node. This is called Dijkstra’s algorithm by the theoretical computer science community, and uniform-cost search Uniform-cost search by the AI community. Implement the Dijkstra’s algorithm to all the cities in India and their Road distances. This info may be taken from open sources.

In [7]:
import random
import heapq
import numpy as np

In [ ]:
def dijkstra(graph, src):
  """
  dijkstra aka uniform cost search
  """
  n = len(graph)
  dist = [float('inf')] * n
  dist[src] = 0
  prev = [None] * n
  pq = [(0,src)]
  while pq:
    d, u = heapq.heappop(pq)
    if d > dist[u]:
      continue
    for v in range(n):
      w = graph[u][v]
      if u != v and w > 0:

        if dist[u] + w < dist[v]:
          dist[v] = dist[u]+w
          prev[v] = u
          heapq.heappush(pq, (dist[v],v))
  return dist, prev

In [ ]:
def dist_path(prev, target):
  path = []
  while target is not None:
    path.append(target)
    target = prev[target]
  return path[::-1]

In [ ]:
cities = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Chennai", "Kolkata"]
city_index = {name: i for i, name in enumerate(cities)}

Cities_Distances = [
    [0, 1419, 2121, 1499, 2107, 1453],      # Delhi
    [1419, 0, 984, 711, 1290, 1939],        # Mumbai
    [2121, 984, 0, 575, 346, 1875],         # Bengaluru
    [1499, 711, 575, 0, 633, 1492],         # Hyderabad
    [2107, 1290, 346, 633, 0, 1684],        # Chennai
    [1453, 1939, 1875, 1492, 1684, 0]       # Kolkata
]

src_city = "Delhi"
distances, prev = dijkstra(Cities_Distances, city_index[src_city])


In [ ]:
for city in cities:
    idx = city_index[city]
    path = [cities[i] for i in dist_path(prev, idx)]
    print(f"{city}: {distances[idx]} km  (path: {' -> '.join(path)})")

Delhi: 0 km  (path: Delhi)
Mumbai: 1419 km  (path: Delhi -> Mumbai)
Bengaluru: 2074 km  (path: Delhi -> Hyderabad -> Bengaluru)
Hyderabad: 1499 km  (path: Delhi -> Hyderabad)
Chennai: 2107 km  (path: Delhi -> Chennai)
Kolkata: 1453 km  (path: Delhi -> Kolkata)


Q2. An Unmanned Ground Vehicle (UGV) is a robot that finds the optimal path from a given user-specified start node to a user-specified Goal node on a map of a small area in a battlefield (Eg 70x70 Kms). There are obstacles known a-priori. The density of the obstacles can be generated randomly with three different levels of density. Design an algorithm that makes the UGV to navigate through this grid space avoiding all the known obstacles to reach the goal by the shortest distance. Trace this path along with the Measures of Effectiveness.

> Battlefield 70 * 70 with 1 representing obstacles and 0 free cells   

In [9]:
def battle_field(size = 70, density_lvl= 'med', seed = None, start=None, goal=None):
  """
  Generates a battle field givee the size with randomly placed objects.
  Parameters:
    size (int): grid is size x size (default 70x70 km)
    density_lvl (str): 'low', 'med', or 'high' obstacle density
    start (tuple: Int): row, col of start cell — guaranteed free if given
    goal (tuple: Int): row, col of goal cell — guaranteed free if given
    seed (int): optional seed for reproducible grids

  Returns:
    2D numpy array, 0 = free, 1 = obstacle
  """
  if seed is not None:
    np.random.seed(seed)

  density = {'low': 0.10, 'med': 0.25, 'high': 0.40}
  probability = density.get(density_lvl.lower(),0.25)

  grid = (np.random.rand(size,size) < probability).astype(int)

  if start is not None:
    grid[start] = 0
  if goal is not None:
    grid[goal] = 0

  return grid


In [10]:
grid = battle_field(70, 'low', start=(0,0), goal=(69,69), seed=7)

> The problem can be solved by using the A*

In [4]:
def _manhattan(a,b):
  """
  utility function that returns the manhattan distance given 2 points
  Manhattan distance gives the  length of the shortest path under
  grid constraints

  Parameter:
  a (tuple: Int): row column pair of point a
  b (tuple: Int): row column pair of point b

  Returns:
    Manhattan distance gives the  length of the shortest path under
  grid constraints
  """
  return abs(a[0]-b[0]) + abs(a[1]-b[1])

In [5]:
def astar(grid, start, goal):
  """
  Given a grid, start, goal this function finds the solution if it exists based on the a* algorithm

  Parameters:
    grid (2D array): grid[row][col] == 0 is free, == 1 is obstacle
    start (tuple: Int): row, column pair of the start cell
    goal (tuple: Int): row, column pair of the goal cell

    Returns:
      list of (row, col) tuples from start to goal, or [] if no path exists

  """
  n_rows, n_cols = len(grid), len(grid[0])
  pq = []
  g_scores = {}
  came_from = {}
  visited = set()

  g_scores[start] = 0
  heapq.heappush(pq, (_manhattan(start,goal), start))

  while pq:
    priority, current = heapq.heappop(pq)

    if current in visited:
     continue
    visited.add(current)

    if current == goal:
      break
    r, c = current
    neighbours = [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]

    for neighbour in neighbours:
      nr, nc = neighbour

      if not (0<=nr<n_rows and 0 <= nc < n_cols):
        continue
      if grid[nr][nc] == 1:
        continue
      temp_g = g_scores[current] + 1

      if neighbour not in g_scores or temp_g < g_scores[neighbour]:
        g_scores[neighbour] = temp_g
        came_from[neighbour] = current
        new_priority = temp_g + _manhattan(neighbour,goal)
        heapq.heappush(pq,(new_priority, neighbour))

  if goal not in came_from and goal != start:
    return []
  path = [goal]
  while path[-1] != start:
      path.append(came_from[path[-1]])
  path.reverse()
  return path

In [25]:
path = astar(grid, start = (0,0), goal=(69,69))

In [26]:
grid

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 1, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [27]:
path

[(0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (0, 6),
 (1, 6),
 (1, 7),
 (1, 8),
 (1, 9),
 (1, 10),
 (1, 11),
 (1, 12),
 (1, 13),
 (1, 14),
 (1, 15),
 (1, 16),
 (1, 17),
 (1, 18),
 (1, 19),
 (1, 20),
 (1, 21),
 (2, 21),
 (2, 22),
 (2, 23),
 (2, 24),
 (2, 25),
 (2, 26),
 (2, 27),
 (3, 27),
 (3, 28),
 (3, 29),
 (3, 30),
 (3, 31),
 (3, 32),
 (3, 33),
 (4, 33),
 (4, 34),
 (5, 34),
 (5, 35),
 (5, 36),
 (5, 37),
 (5, 38),
 (5, 39),
 (5, 40),
 (5, 41),
 (5, 42),
 (5, 43),
 (5, 44),
 (5, 45),
 (5, 46),
 (5, 47),
 (5, 48),
 (5, 49),
 (5, 50),
 (5, 51),
 (5, 52),
 (5, 53),
 (5, 54),
 (6, 54),
 (7, 54),
 (7, 55),
 (8, 55),
 (9, 55),
 (9, 56),
 (9, 57),
 (9, 58),
 (9, 59),
 (10, 59),
 (10, 60),
 (11, 60),
 (12, 60),
 (13, 60),
 (13, 61),
 (13, 62),
 (14, 62),
 (15, 62),
 (16, 62),
 (17, 62),
 (18, 62),
 (19, 62),
 (20, 62),
 (21, 62),
 (22, 62),
 (23, 62),
 (24, 62),
 (25, 62),
 (26, 62),
 (27, 62),
 (28, 62),
 (29, 62),
 (30, 62),
 (31, 62),
 (32, 62),
 (33, 62),
 (34, 62),
 (35, 62),
 (

In [28]:
len(path) > 0

True


Q3. In the above problem, we relax the condition that all obstacles are Static. In a real world, obstacles can be dynamic and not known a priori. How do you make the UGV navigate and find the optinal path in a dynamic obstacles environment.